# LeetCode 2099 — Find Subsequence of Length K With the Largest Sum bằng Python

Link đề: https://leetcode.com/problems/find-subsequence-of-length-k-with-the-largest-sum/ — Độ khó: **Easy**

Skeleton LeetCode:

```python
class Solution:
    def maxSubsequence(self, nums: List[int], k: int) -> List[int]:
        ...
```

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu **subsequence** (dãy con) khác gì **subarray** và khác gì "chọn k số lớn nhất".
2. Biết dùng `sorted(..., key=...)`, `enumerate`, `heapq.nlargest` và `Counter`/đếm ngưỡng.
3. Viết 4 cách: brute force tổ hợp, sort theo chỉ số, heap, và ngưỡng phần tử thứ k.
4. Nắm insight: tổng lớn nhất = k số lớn nhất, sau đó chỉ cần **trả lại đúng thứ tự gốc**.
5. Biết kiểm tra kết quả bằng hàm checker (vì đáp án có thể không duy nhất khi có số trùng).

## 2. Đề bài & ví dụ

Cho mảng nguyên `nums` và số nguyên `k`. Hãy chọn ra một **dãy con** (subsequence) độ dài `k` có **tổng lớn nhất**. Dãy con giữ nguyên thứ tự tương đối của các phần tử trong `nums`. Trả về dãy con đó (nếu có nhiều đáp án, trả về bất kỳ đáp án nào).

| nums | k | kết quả |
|---|---|---|
| `[2,1,3,3]` | 2 | `[3,3]` |
| `[-1,-2,3,4]` | 3 | `[-1,3,4]` |
| `[3,4,3,3]` | 2 | `[3,4]` (hoặc `[4,3]`) |

Constraints: `1 <= nums.length <= 1000`, `-10^5 <= nums[i] <= 10^5`, `1 <= k <= nums.length`.

Ví dụ 3 cho thấy đáp án **không duy nhất**: có nhiều số `3` bằng nhau, chọn số 3 nào cũng cùng tổng 7.

In [1]:
examples = [([2, 1, 3, 3], 2), ([-1, -2, 3, 4], 3), ([3, 4, 3, 3], 2)]
for nums, k in examples:
    print("nums =", nums, "| k =", k, "| k số lớn nhất =", sorted(nums, reverse=True)[:k])

nums = [2, 1, 3, 3] | k = 2 | k số lớn nhất = [3, 3]
nums = [-1, -2, 3, 4] | k = 3 | k số lớn nhất = [4, 3, -1]
nums = [3, 4, 3, 3] | k = 2 | k số lớn nhất = [4, 3]


## 3. Kiến thức nền

### 3.1. Subsequence và subarray

- **Subarray**: các phần tử *liền nhau*, ví dụ `[2,1]` của `[2,1,3,3]`.
- **Subsequence**: được *bỏ qua* phần tử nhưng **giữ thứ tự**, ví dụ `[2,3]` (bỏ `1`), nhưng `[3,2]` thì không.

In [2]:
def is_subsequence(small, big):
    it = iter(big)
    return all(x in it for x in small)

print(is_subsequence([2, 3], [2, 1, 3, 3]))
print(is_subsequence([3, 2], [2, 1, 3, 3]))

True
False


Mẹo `x in it` với `it = iter(big)`: mỗi lần kiểm tra, iterator **tiêu thụ** các phần tử đã đi qua, nên các phần tử của `small` buộc phải xuất hiện theo đúng thứ tự.

### 3.2. `sorted` với `key`, `reverse` và sort các chỉ số

`sorted(range(n), key=lambda i: nums[i])` sắp xếp **chỉ số** theo giá trị tương ứng. `reverse=True` để giảm dần. Python sort là **ổn định**: hai phần tử bằng nhau giữ nguyên thứ tự ban đầu.

In [3]:
nums = [2, 1, 3, 3]
idx = sorted(range(len(nums)), key=lambda i: nums[i], reverse=True)
print("chỉ số xếp theo giá trị giảm dần:", idx)
print("lấy 2 chỉ số đầu rồi sort lại:", sorted(idx[:2]))

chỉ số xếp theo giá trị giảm dần: [2, 3, 0, 1]
lấy 2 chỉ số đầu rồi sort lại: [2, 3]


### 3.3. `heapq.nlargest`

`heapq.nlargest(k, iterable, key=...)` trả về `k` phần tử lớn nhất theo `key`. Dùng heap bên trong nên thời gian khoảng `O(n log k)`. Ta có thể đưa vào cặp `(giá trị, chỉ số)` để nhớ vị trí.

In [4]:
import heapq
nums = [3, 4, 3, 3]
print(heapq.nlargest(2, nums))
print(heapq.nlargest(2, [(v, i) for i, v in enumerate(nums)]))

[4, 3]
[(4, 1), (3, 3)]


### 3.4. `itertools.combinations`

`combinations(a, k)` sinh mọi tổ hợp `k` phần tử (theo **chỉ số tăng**), giữ đúng thứ tự gốc — chính là mọi subsequence độ dài `k`. Số tổ hợp là `C(n, k)`, rất lớn khi `n` lớn, nên chỉ dùng làm brute force kiểm chứng.

In [5]:
from itertools import combinations
print(list(combinations([2, 1, 3, 3], 2)))
print(len(list(combinations(range(10), 3))))

[(2, 1), (2, 3), (2, 3), (1, 3), (1, 3), (3, 3)]
120


## 4. Ý tưởng nền tảng — tính bằng tay

`nums = [-1, -2, 3, 4]`, `k = 3`. Muốn tổng lớn nhất thì chọn 3 số **lớn nhất**: `4, 3, -1` (tổng 6). Tại sao không phải `-2`? Vì `-1 > -2`. Giá trị của các phần tử không phụ thuộc nhau, nên greedy "lấy k số lớn nhất" cho tổng tối ưu.

Còn thứ tự? Đề bắt giữ thứ tự gốc: trong `nums` thì `-1` đứng trước `3`, `3` đứng trước `4` → `[-1, 3, 4]`.

In [6]:
nums, k = [-1, -2, 3, 4], 3
top = sorted(nums, reverse=True)[:k]
print("k số lớn nhất:", top, "| tổng =", sum(top))
print("nhưng thứ tự gốc là:", [x for x in nums if x in top])

k số lớn nhất: [4, 3, -1] | tổng = 6
nhưng thứ tự gốc là: [-1, 3, 4]


## 5. Cách 1 — Brute Force: thử mọi dãy con

Dùng `combinations` thử mọi dãy con độ dài `k`, giữ cái có tổng lớn nhất.

In [7]:
from itertools import combinations

def sub_brute(nums, k):
    best, best_sum = None, None
    for comb in combinations(nums, k):
        s = sum(comb)
        if best_sum is None or s > best_sum:
            best, best_sum = list(comb), s
    return best

for nums, k in examples:
    print(sub_brute(nums, k))

[3, 3]
[-1, 3, 4]
[3, 4]


Độ phức tạp: có `C(n, k)` tổ hợp, mỗi cái tính tổng `O(k)` → **Time `O(C(n,k) * k)`**, bùng nổ tổ hợp (n = 1000 là không thể). Nút thắt: ta thử cả những tập chắc chắn không tối ưu.

## 6. Ý tưởng thuật toán / bước đột phá

- Tổng của `k` số **chỉ phụ thuộc vào tập giá trị**, không phụ thuộc thứ tự → tổng lớn nhất khi chọn `k` số lớn nhất (đa tập).
- Quy tắc "nếu… thì…": nếu một số nằm trong top-`k` thì **giữ chỉ số** của nó; cuối cùng sắp các chỉ số tăng dần để phục hồi thứ tự gốc.
- Khi có số trùng với phần tử thứ `k`, chọn *bất kỳ* bản nào cũng được — nhưng phải chọn đúng **số lượng**.

Invariant: tập chỉ số được chọn có đúng `k` phần tử và không có phần tử bị bỏ nào lớn hơn phần tử được chọn nào.

## 7. Các cách tiếp theo

### Cách 2 — Sort chỉ số theo giá trị giảm dần

Sắp mọi chỉ số theo `nums[i]` giảm dần, lấy `k` chỉ số đầu, sort lại các chỉ số đó tăng dần, rồi đọc giá trị.

In [8]:
def sub_sort_idx(nums, k):
    idx = sorted(range(len(nums)), key=lambda i: nums[i], reverse=True)
    chosen = sorted(idx[:k])
    return [nums[i] for i in chosen]

for nums, k in examples:
    print(sub_sort_idx(nums, k))

[3, 3]
[-1, 3, 4]
[3, 4]


In [9]:
from typing import List

class SolutionSortIdx:
    def maxSubsequence(self, nums: List[int], k: int) -> List[int]:
        idx = sorted(range(len(nums)), key=lambda i: nums[i], reverse=True)
        return [nums[i] for i in sorted(idx[:k])]

print(SolutionSortIdx().maxSubsequence([-1, -2, 3, 4], 3))

[-1, 3, 4]


Độ phức tạp: **Time `O(n log n)`**, **Space `O(n)`**.

### Cách 3 — Heap: `heapq.nlargest` trên cặp `(giá trị, chỉ số)`

Lấy `k` cặp lớn nhất, sort theo chỉ số, đọc giá trị. Heap kích thước `k` nên thường nhanh hơn khi `k` nhỏ.

In [10]:
import heapq

def sub_heap(nums, k):
    top = heapq.nlargest(k, ((v, i) for i, v in enumerate(nums)))
    top.sort(key=lambda p: p[1])
    return [v for v, i in top]

for nums, k in examples:
    print(sub_heap(nums, k))

[3, 3]
[-1, 3, 4]
[4, 3]


In [11]:
from typing import List
import heapq

class SolutionHeap:
    def maxSubsequence(self, nums: List[int], k: int) -> List[int]:
        top = heapq.nlargest(k, ((v, i) for i, v in enumerate(nums)))
        top.sort(key=lambda p: p[1])
        return [v for v, _ in top]

print(SolutionHeap().maxSubsequence([2, 1, 3, 3], 2))

[3, 3]


Độ phức tạp: **Time `O(n log k + k log k)`**, **Space `O(k)`** (không tính input).

### Cách 4 — Ngưỡng phần tử thứ k (không cần sort chỉ số)

1. Sort bản sao giảm dần, lấy `th = sorted_desc[k-1]` (phần tử thứ `k`).
2. Số phần tử `> th` là `big`. Phải lấy thêm `need = k - big` phần tử **bằng** `th`.
3. Quét `nums` theo thứ tự gốc: lấy `x > th`; lấy `x == th` nếu còn `need > 0`.

Kết quả tự động đúng thứ tự gốc, không cần sort lại chỉ số.

In [12]:
def sub_threshold(nums, k):
    th = sorted(nums, reverse=True)[k - 1]
    big = sum(1 for x in nums if x > th)
    need = k - big
    res = []
    for x in nums:
        if x > th:
            res.append(x)
        elif x == th and need > 0:
            res.append(x)
            need -= 1
    return res

for nums, k in examples:
    print(sub_threshold(nums, k))

[3, 3]
[-1, 3, 4]
[3, 4]


In [13]:
from typing import List

class SolutionThreshold:
    def maxSubsequence(self, nums: List[int], k: int) -> List[int]:
        th = sorted(nums, reverse=True)[k - 1]
        need = k - sum(1 for x in nums if x > th)
        res = []
        for x in nums:
            if x > th:
                res.append(x)
            elif x == th and need > 0:
                res.append(x)
                need -= 1
        return res

print(SolutionThreshold().maxSubsequence([3, 4, 3, 3], 2))

[3, 4]


Độ phức tạp: **Time `O(n log n)`** do sort bản sao (có thể xuống `O(n)` kỳ vọng bằng quickselect), **Space `O(n)`**.

## 8. TRACE từng bước

Trace Cách 4 với `nums = [3, 4, 3, 3]`, `k = 2`. Sort giảm dần `[4, 3, 3, 3]` → `th = 3`. Chỉ có `4 > 3` nên `big = 1`, `need = 1`.

| x | điều kiện | need | res |
|---|---|---|---|
| 3 | `x == th`, need=1 → lấy | 0 | `[3]` |
| 4 | `x > th` → lấy | 0 | `[3,4]` |
| 3 | `x == th`, need=0 → bỏ | 0 | `[3,4]` |
| 3 | `x == th`, need=0 → bỏ | 0 | `[3,4]` |

In [14]:
def trace_threshold(nums, k):
    th = sorted(nums, reverse=True)[k - 1]
    big = sum(1 for x in nums if x > th)
    need = k - big
    print(f"th={th}, big={big}, need={need}")
    res = []
    for i, x in enumerate(nums):
        if x > th:
            res.append(x)
            act = "lấy (lớn hơn ngưỡng)"
        elif x == th and need > 0:
            res.append(x)
            need -= 1
            act = "lấy (bằng ngưỡng)"
        else:
            act = "bỏ"
        print(f"i={i}, x={x}: {act}, need={need}, res={res}")
    return res

trace_threshold([3, 4, 3, 3], 2)

th=3, big=1, need=1
i=0, x=3: lấy (bằng ngưỡng), need=0, res=[3]
i=1, x=4: lấy (lớn hơn ngưỡng), need=0, res=[3, 4]
i=2, x=3: bỏ, need=0, res=[3, 4]
i=3, x=3: bỏ, need=0, res=[3, 4]


[3, 4]

In [15]:
def trace_sort_idx(nums, k):
    idx = sorted(range(len(nums)), key=lambda i: nums[i], reverse=True)
    print("chỉ số theo giá trị giảm dần:", idx, "->", [nums[i] for i in idx])
    chosen = sorted(idx[:k])
    print("k chỉ số đầu:", idx[:k], "-> sort lại:", chosen)
    return [nums[i] for i in chosen]

print(trace_sort_idx([-1, -2, 3, 4], 3))

chỉ số theo giá trị giảm dần: [3, 2, 0, 1] -> [4, 3, -1, -2]
k chỉ số đầu: [3, 2, 0] -> sort lại: [0, 2, 3]
[-1, 3, 4]


## 9. Code nộp LeetCode

In [16]:
from typing import List

class Solution:
    def maxSubsequence(self, nums: List[int], k: int) -> List[int]:
        idx = sorted(range(len(nums)), key=lambda i: nums[i], reverse=True)
        return [nums[i] for i in sorted(idx[:k])]

print(Solution().maxSubsequence([2, 1, 3, 3], 2))

[3, 3]


## 10. Vì sao thuật toán đúng?

**Tối ưu về tổng.** Giả sử một đáp án có phần tử `a` được chọn và phần tử `b` không được chọn mà `b > a`. Đổi `a` thành `b` làm tổng tăng, vậy đáp án đó không tối ưu. Do đó trong đáp án tối ưu, mọi phần tử bị bỏ đều `<=` mọi phần tử được chọn — tức tập chọn chính là `k` số lớn nhất (đa tập).

**Đúng thứ tự.** Một dãy con là *tập chỉ số tăng dần* ánh xạ sang giá trị. Sau khi chọn đúng tập chỉ số, sort chỉ số tăng dần cho ra đúng dãy con hợp lệ.

**Về số trùng.** Nếu giá trị thứ `k` xuất hiện nhiều lần, chọn bất kỳ bản nào cũng cùng tổng, nên đáp án không duy nhất nhưng đều tối ưu.

## 11. Phân tích Time / Space Complexity

Gọi `n = len(nums)`.

| Cách | Time | Space |
|---|---|---|
| 1. Brute combinations | `O(C(n,k)·k)` | `O(k)` |
| 2. Sort chỉ số | `O(n log n)` | `O(n)` |
| 3. Heap nlargest | `O(n log k)` | `O(k)` |
| 4. Ngưỡng phần tử thứ k | `O(n log n)` (hoặc `O(n)` kỳ vọng với quickselect) | `O(n)` |

Với `n ≤ 1000` cả ba cách tối ưu chạy tức thì. Worst case và average đều như bảng.

## 12. Bộ test

Vì đáp án không duy nhất, ta **không** so sánh bằng `==` mà dùng hàm `check(nums, k, res)`: `res` phải (a) có độ dài `k`, (b) là subsequence của `nums`, (c) có tổng bằng tổng của `k` số lớn nhất.

In [17]:
def check(nums, k, res):
    best = sum(sorted(nums, reverse=True)[:k])
    return len(res) == k and is_subsequence(res, nums) and sum(res) == best

impls = [sub_brute, sub_sort_idx, sub_heap, sub_threshold,
         Solution().maxSubsequence, SolutionSortIdx().maxSubsequence,
         SolutionHeap().maxSubsequence, SolutionThreshold().maxSubsequence]

cases = [
    ([2, 1, 3, 3], 2), ([-1, -2, 3, 4], 3), ([3, 4, 3, 3], 2),
    ([5], 1), ([-5], 1), ([1, 1, 1, 1], 3), ([-3, -2, -1], 3),
    ([-3, -2, -1], 1), ([0, 0, 0], 2), ([10, -10, 10, -10], 2),
]
for nums, k in cases:
    for f in impls:
        res = f(list(nums), k)
        assert check(nums, k, res), (f.__name__, nums, k, res)
print("Đã kiểm", len(cases), "ca x", len(impls), "cách: tất cả hợp lệ")

Đã kiểm 10 ca x 8 cách: tất cả hợp lệ


In [18]:
# vài kết quả cụ thể (không có số trùng ở ngưỡng nên đáp án duy nhất)
assert Solution().maxSubsequence([-1, -2, 3, 4], 3) == [-1, 3, 4]
assert Solution().maxSubsequence([2, 1, 3, 3], 2) == [3, 3]
assert Solution().maxSubsequence([5], 1) == [5]
print("ví dụ đề: OK")

ví dụ đề: OK


In [19]:
import random
random.seed(2099)
for _ in range(1500):
    n = random.randint(1, 9)
    nums = [random.randint(-6, 6) for _ in range(n)]
    k = random.randint(1, n)
    for f in impls:
        assert check(nums, k, f(list(nums), k)), (f.__name__, nums, k)
print("1500 input ngẫu nhiên (n <= 9) đối chiếu brute force: OK")

for _ in range(300):
    n = random.randint(1, 200)
    nums = [random.randint(-100, 100) for _ in range(n)]
    k = random.randint(1, n)
    for f in impls[1:4]:
        assert check(nums, k, f(list(nums), k))
print("300 input lớn hơn: các cách tối ưu hợp lệ")

1500 input ngẫu nhiên (n <= 9) đối chiếu brute force: OK


300 input lớn hơn: các cách tối ưu hợp lệ


## 13. Các lỗi người mới hay gặp

### Lỗi 1: Trả về k số lớn nhất theo thứ tự sort (mất thứ tự gốc)

`sorted(nums, reverse=True)[:k]` có đúng tổng nhưng **không phải subsequence** của `nums`.

In [20]:
def wrong1(nums, k):
    return sorted(nums, reverse=True)[:k]

nums, k = [-1, -2, 3, 4], 3
r = wrong1(nums, k)
print(r, "sum đúng:", sum(r) == 6, "| là subsequence:", is_subsequence(r, nums))

[4, 3, -1] sum đúng: True | là subsequence: False


### Lỗi 2: Dùng `x in top` để lọc → lấy thừa phần tử trùng

Với `nums = [3,4,3,3]`, `k = 2`, top = `[4,3]`. Lọc `[x for x in nums if x in top]` lấy *cả ba* số 3 và số 4 → độ dài 4 chứ không phải 2.

In [21]:
def wrong2(nums, k):
    top = sorted(nums, reverse=True)[:k]
    return [x for x in nums if x in top]

print(wrong2([3, 4, 3, 3], 2), "-> độ dài", len(wrong2([3, 4, 3, 3], 2)), "(phải là 2)")

[3, 4, 3, 3] -> độ dài 4 (phải là 2)


### Lỗi 3: Sort các cặp `(giá trị, chỉ số)` rồi quên sort lại theo chỉ số

Lấy top-k theo giá trị xong phải sort theo **chỉ số** trước khi đọc giá trị.

In [22]:
nums, k = [-1, -2, 3, 4], 3
pairs = sorted(((v, i) for i, v in enumerate(nums)), reverse=True)[:k]
print("sai (chưa sort theo chỉ số):", [v for v, i in pairs])
pairs.sort(key=lambda p: p[1])
print("đúng:", [v for v, i in pairs])

sai (chưa sort theo chỉ số): [4, 3, -1]
đúng: [-1, 3, 4]


### Lỗi 4: `sorted(nums)[:k]` (lấy k số nhỏ nhất)

Nhầm hướng sort cho ra tổng nhỏ nhất thay vì lớn nhất.

In [23]:
nums, k = [2, 1, 3, 3], 2
print("sai (k nhỏ nhất):", sorted(nums)[:k], sum(sorted(nums)[:k]))
print("đúng:", sorted(nums, reverse=True)[:k], sum(sorted(nums, reverse=True)[:k]))

sai (k nhỏ nhất): [1, 2] 3
đúng: [3, 3] 6


## 14. Mẹo làm bài nhanh & nhận dạng pattern

- "Chọn `k` phần tử để tổng lớn nhất, **giữ thứ tự**" → chọn k lớn nhất theo *giá trị*, sau đó khôi phục thứ tự bằng *chỉ số*.
- Pattern **"top-k + giữ vị trí"**: gắn chỉ số vào dữ liệu `(giá trị, chỉ số)` rồi sort/heap.
- Khi đáp án không duy nhất, viết checker (độ dài + subsequence + tổng) để tự test.
- `heapq.nlargest(k, ...)` thích hợp khi `k` nhỏ; sort thẳng khi `k` gần `n`.
- Ngưỡng + đếm số lượng bằng ngưỡng là kỹ thuật xử lý trùng gọn gàng, hay xuất hiện ở các bài top-k.

## 15. Biến thể của bài

1. Dãy con độ dài `k` có **tổng nhỏ nhất**: đảo hướng (k số nhỏ nhất).
2. Yêu cầu đáp án **nhỏ nhất theo thứ tự từ điển** trong các đáp án có cùng tổng (bài tương tự: "Find the Most Competitive Subsequence", cần stack).
3. Subarray độ dài `k` có tổng lớn nhất (cửa sổ trượt, bài khác hẳn).
4. Chọn tối đa `k` phần tử nhưng tổng lớn nhất: chỉ lấy số dương.
5. Trả về **chỉ số** thay vì giá trị.

In [24]:
def min_sum_subsequence(nums, k):
    idx = sorted(range(len(nums)), key=lambda i: nums[i])
    return [nums[i] for i in sorted(idx[:k])]

def top_k_indices(nums, k):
    idx = sorted(range(len(nums)), key=lambda i: nums[i], reverse=True)
    return sorted(idx[:k])

print(min_sum_subsequence([2, 1, 3, 3], 2))
print(top_k_indices([-1, -2, 3, 4], 3))

[2, 1]
[0, 2, 3]


## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Brute combinations | thử mọi dãy con | `O(C(n,k)·k)` | `O(k)` | chỉ để kiểm chứng, `n` nhỏ |
| 2. Sort chỉ số | top-k theo giá trị, sort lại chỉ số | `O(n log n)` | `O(n)` | nộp bài, dễ nhất |
| 3. Heap nlargest | heap kích thước k | `O(n log k)` | `O(k)` | `k` nhỏ so với `n` |
| 4. Ngưỡng thứ k | lọc theo ngưỡng + đếm số bằng ngưỡng | `O(n log n)` | `O(n)` | không muốn sort chỉ số |

## 17. 5 bài tập vận dụng

**Bài 1 (đoán output):** `sub_sort_idx([1, 5, 2, 5], 2)` trả về gì?

**Bài 2 (tính tay):** `nums = [4, -1, 4, 2]`, `k = 3`. Tìm ngưỡng `th`, `need` và dãy con theo Cách 4.

**Bài 3 (điền TODO):** hoàn thành `exercise_heap` dùng `heapq.nlargest`.

**Bài 4 (tự viết lại):** viết `exercise_threshold` không nhìn lại code.

**Bài 5 (mini challenge):** viết `exercise_min_sum(nums, k)` trả về dãy con độ dài `k` có tổng **nhỏ nhất** (giữ thứ tự).

In [25]:
# Bài 1: chạy để so với dự đoán
print(sub_sort_idx([1, 5, 2, 5], 2))

[5, 5]


In [26]:
def exercise_heap(nums, k):
    # TODO: top = heapq.nlargest(k, ...) trên cặp (giá trị, chỉ số)
    # TODO: sort top theo chỉ số rồi trả về danh sách giá trị
    pass

def exercise_threshold(nums, k):
    # TODO: th = phần tử thứ k trong danh sách giảm dần; đếm need; quét nums
    pass

def exercise_min_sum(nums, k):
    # TODO: tương tự Cách 2 nhưng sort tăng dần
    pass

# (không gọi khi chưa làm xong)

## 18. Gợi ý 3 mức

| Bài | Mức 1 | Mức 2 | Mức 3 |
|---|---|---|---|
| 1 | Hai số lớn nhất là hai số 5 | Chúng ở chỉ số 1 và 3 | `[5, 5]` |
| 2 | Sort giảm dần: `[4,4,2,-1]` | Phần tử thứ 3 là `2` → `th = 2`, chỉ có hai số 4 lớn hơn | `need = 1`, kết quả `[4, 4, 2]` |
| 3 | Dùng `enumerate(nums)` để tạo cặp | `top.sort(key=lambda p: p[1])` | `[v for v, _ in top]` |
| 4 | Nhớ ba bước: ngưỡng, `need`, quét | Chỉ lấy `x == th` khi `need > 0` | Xem Cách 4 ở mục 7 |
| 5 | Sort chỉ số theo giá trị tăng dần | Lấy `k` chỉ số đầu | Sort lại chỉ số, đọc giá trị |

## 19. Đáp án tham khảo cho bài tập

Bài 1: `[5, 5]`. Bài 2: `th = 2`, `need = 1`, kết quả `[4, 4, 2]`.

In [27]:
import heapq

def exercise_heap(nums, k):
    top = heapq.nlargest(k, ((v, i) for i, v in enumerate(nums)))
    top.sort(key=lambda p: p[1])
    return [v for v, _ in top]

def exercise_threshold(nums, k):
    th = sorted(nums, reverse=True)[k - 1]
    need = k - sum(1 for x in nums if x > th)
    res = []
    for x in nums:
        if x > th:
            res.append(x)
        elif x == th and need > 0:
            res.append(x)
            need -= 1
    return res

def exercise_min_sum(nums, k):
    idx = sorted(range(len(nums)), key=lambda i: nums[i])
    return [nums[i] for i in sorted(idx[:k])]

assert sub_sort_idx([1, 5, 2, 5], 2) == [5, 5]
assert exercise_threshold([4, -1, 4, 2], 3) == [4, 4, 2]
assert exercise_heap([-1, -2, 3, 4], 3) == [-1, 3, 4]
assert check([3, 4, 3, 3], 2, exercise_heap([3, 4, 3, 3], 2))
assert exercise_min_sum([2, 1, 3, 3], 2) == [2, 1]
assert exercise_min_sum([5, 9, 1], 1) == [1]
print("Đáp án bài tập đều đúng")

Đáp án bài tập đều đúng


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template (top-k giữ thứ tự):**

```python
idx = sorted(range(len(nums)), key=lambda i: nums[i], reverse=True)
return [nums[i] for i in sorted(idx[:k])]
```

**Checklist:**

- [ ] Chọn k số **lớn nhất** (không phải nhỏ nhất)?
- [ ] Kết quả giữ **thứ tự gốc** (sort lại chỉ số)?
- [ ] Độ dài đúng `k` kể cả khi có số trùng?
- [ ] Đã thử toàn số âm, `k = len(nums)`, `k = 1`?
- [ ] Test bằng checker vì đáp án không duy nhất?